In [1]:
from requests.adapters import HTTPAdapter 
from urllib3.util.retry import Retry
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

In [4]:
all_listings = []
base_url = 'https://propertypro.ng/property-for-rent/in/abuja/?search=Abuja&auto=Abuja&page='

# --- IMPROVEMENT 1: Define User Agents outside loop ---
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36"
]

# --- IMPROVEMENT 2: Instantiate Session ONCE outside the loop ---
# This maintains cookies and TCP connections, making you look like a real user browsing multiple pages.
session = requests.Session()
retries = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))
session.headers.update({"User-Agent": random.choice(user_agents)})

for page in range(1, 105):
    print(f"Scraping page {page}...") # Added print to track progress
    url = f"{base_url}{page}"
    
    try:
        # --- IMPROVEMENT 3: Add 'Referer' header ---
        # This makes it look like you clicked 'Next Page' from the previous page
        headers = {"Referer": base_url + str(page - 1) if page > 1 else "https://propertypro.ng"}
        
        response = session.get(url, headers=headers, timeout=15)
        
        # Check if the page loaded correctly
        if response.status_code != 200:
            print(f"Warning: Failed to load page {page}. Status code: {response.status_code}")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')
        listings = soup.select(".property-listing")
        
        # Stop loop if page is empty (reached end of data)
        if not listings:
            print("No more listings found.")
            break

        for listing in listings:
            try: # --- IMPROVEMENT 4: Nested try/except ---
                # This ensures one bad listing doesn't crash the whole script
                price = listing.find('div', class_ = 'pl-price').find('h3').text.strip()
                location = listing.find('div', class_ = 'pl-title').find("p").text.strip()
                bedrooms = listing.find('div', class_ = 'pl-title').find_all('a')[0].text.strip()
                list_date = listing.find('p', class_ = 'date-added').text.strip()


                all_listings.append({
                    'page': page,
                    'Price': price,
                    'Location': location,
                    'Bedrooms': bedrooms,
                    'List Date': list_date})
                #all_listings.append(listing_info)
            except Exception as e:
                # If a specific house fails, skip it but print error
                # print(f"Error parsing a listing: {e}") 
                continue

        # --- IMPROVEMENT 5: Randomized Sleep ---
        # Random sleep between 3 to 7 seconds helps avoid bot detection patterns
        time.sleep(random.uniform(3, 7))

        # --- IMPROVEMENT 6: Batch Save ---
        # Save every 10 pages instead of every single page to reduce disk usage while keeping data safe
        if page % 10 == 0:
            df = pd.DataFrame(all_listings)
            df.to_csv(r"C:\Users\abba\Desktop\data science stuff\PROJECTS2\Abuja-District-Rental-Insights\data\raw\ppro_raw_listings.csv", index=False)
            print(f"checkpoint: saved data up to page {page}")

    except Exception as e:
        print(f"Critical error on page {page}: {e}")
        time.sleep(10) # Wait longer if a network error occurs

# Final save at the very end
df = pd.DataFrame(all_listings)
df.to_csv(r"C:\Users\abba\Desktop\data science stuff\PROJECTS2\Abuja-District-Rental-Insights\data\raw\ppro_raw_listings.csv", index=False)
print("Scraping completed successfully.")

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
checkpoint: saved data up to page 10
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
checkpoint: saved data up to page 20
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
checkpoint: saved data up to page 30
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
checkpoint: saved data up to page 40
Scraping page 41...
Scraping page 42...
Scraping page 43...
S